# Agentic AI Fundamentals

## Notebook 00: From Model Output to Controlled Action

Agentic AI is often introduced through frameworks, diagrams, and impressive demonstrations. That approach can make the subject appear simpler than it really is because the framework hides the decisions that matter most: who chooses an action, who is allowed to execute it, where intermediate state is stored, and how the system knows when to stop.

This notebook begins one layer below those frameworks. We will use ordinary Python to construct the smallest useful mental model of an agentic system. There is no model API and no hidden automation yet. By making every transition explicit, we can see which responsibilities belong to the language model and which responsibilities must remain under the control of the surrounding application.

By the end of the notebook, you should be able to explain the following statement precisely:

> A language model may propose an action, but trusted application code validates and executes that action.

### The core idea

A useful way to think about an agentic system is:

> **LLM = proposes what to do**  
>
> **Tools = provide capabilities**  
>
> **Application = decides what is allowed and executes it**  
>
> **Observation = tells the system what happened**

The important distinction is that **proposing an action is not the same as executing it**. The model can suggest what should happen next, while trusted application code remains in control of what actually happens.

### Models, assistants, and agentic systems

Names such as **GPT, Claude, and Gemini** are often associated with AI agents, but a model and an agent are not the same thing.

> **Model** → the underlying intelligence that processes input and generates output  
>
> **Assistant / Product** → an application built around a model, often with tools and additional capabilities  
>
> **Agentic system** → a system where a model can choose actions, use tools, observe results, and continue toward a goal

A simple mental model:
```

    GPT / Claude / Gemini
              │
              │  Model
              ▼
    +---------------------+
    | Instructions        |
    | Tools               |
    | State / Memory      |
    | Control Loop        |
    | Safety Rules        |
    +---------------------+
              │
              ▼
       Agentic System
```

A model alone is therefore **not necessarily an agent**. Agentic behavior comes from the surrounding system that allows the model to make decisions and interact with external capabilities.

## Learning objectives and prerequisites

This course assumes that you can read basic Python functions, dictionaries, conditionals, and exceptions. It does not assume previous experience with language-model APIs, tool calling, RAG, LangChain, LangGraph, or multi-agent systems.

After completing this notebook, you should be able to:

1. distinguish a language-model response from an application action;
2. describe a tool as a controlled application capability rather than a special AI primitive;
3. represent a tool request as structured data;
4. use a registry as an explicit allowlist of executable tools;
5. turn a tool result into an observation that can inform the next decision; and
6. identify the minimum safety and termination controls required by an agent loop.

We deliberately postpone real model calls. A remote model would introduce authentication, cost, latency, and nondeterministic output before we have understood the architecture that surrounds it.

## 1. Begin with a clean execution environment

A notebook executes code inside a Python kernel, and that kernel may not be the same Python environment that your terminal uses. When a package appears to be installed but cannot be imported, the underlying cause is often an environment mismatch rather than a problem with the package itself.

The next cell records the interpreter used by this notebook. Keeping this information visible is a small but useful reproducibility habit, especially once the project begins to depend on external libraries.

In [ ]:
import platform
import sys

print(f"Python version    : {platform.python_version()}")
print(f"Python executable : {sys.executable}")
print(f"Operating system  : {platform.system()} {platform.release()}")

Python version    : 3.13.15
Python executable : /usr/bin/python3
Operating system  : Linux 6.6.122+


### A brief history: from rules to language models

Conversational software did not begin with LLMs. Early systems often used rules written by developers: inspect the input, match a pattern, and choose or transform a response. **ELIZA** (1966), created by Joseph Weizenbaum, is a well-known example. Its DOCTOR script used keyword and pattern matching with substitution rules to produce replies that could resemble a conversation. ELIZA was a rule-based chatbot, not a language model.

Later and simpler applications can use **regular expressions (regex)** to recognize text patterns such as a date or an order number and trigger a predefined response. Regex is one possible technique for implementing rules; it is not a separate stage that every chatbot passed through. Modern LLMs generate responses using learned patterns rather than a small set of handwritten conversation rules.

| Approach | How a response is chosen | Example limitation |
| --- | --- | --- |
| Rule-based chatbot (ELIZA) | Matches input patterns and applies authored reply rules | Struggles when no rule fits |
| Regex-driven application | Matches specified text forms and follows application logic | A differently phrased request may not match |
| Modern language model | Generates output from learned representations and the provided context | Still cannot perform a private action unless the application grants a controlled capability |

These systems differ greatly inside. The useful connection for this notebook is their **application boundary**: the program receives an input and obtains an output. The next section asks what happens when an application needs to act on that output.


## 2. A language model produces output; an application performs work

At the boundary of an application, a language model behaves like a component that receives context and returns a response. The internal computation is sophisticated, but the external contract is still an input-output interaction. If the application sends a collection of messages, the model returns text or structured output based on those messages.

A model does not automatically inherit the capabilities of the computer that called it. It cannot query a private database, inspect a local file, send an email, or invoke a Python function unless the surrounding application deliberately exposes and governs those capabilities.

The function below is not a real language model. It is a deterministic stand-in that lets us examine the application boundary without introducing API-specific details. Its interface resembles a very small model: text enters and text leaves.

In [ ]:
def deterministic_model(user_message: str) -> str:
    """Return a predefined textual response for a few known inputs."""

    normalized_message = user_message.casefold().strip()

    if "capital of romania" in normalized_message:
        return "The capital of Romania is Bucharest."

    if normalized_message in {"hello", "hi"}:
        return "Hello. How can I help?"

    return "I do not have enough information to answer that request."


questions = [
    "Hello",
    "What is the capital of Romania?",
    "What is 145 multiplied by 28?",
]

for question in questions:
    print(f"User  : {question}")
    print(f"Model : {deterministic_model(question)}")
    print("-" * 70)

User  : Hello
Model : Hello. How can I help?
----------------------------------------------------------------------
User  : What is the capital of Romania?
Model : The capital of Romania is Bucharest.
----------------------------------------------------------------------
User  : What is 145 multiplied by 28?
Model : I do not have enough information to answer that request.
----------------------------------------------------------------------


The arithmetic request receives an unhelpful response because the component has neither arithmetic knowledge nor access to a calculator. This is not yet an agentic failure; it is simply a capability boundary. We could expand the function with more hard-coded rules, but that would still leave us with a function whose behavior is fixed by the developer.

An agentic design becomes relevant when the appropriate next action cannot be completely predetermined and a model must choose among available capabilities at runtime. Before we discuss that decision, we need to define what those capabilities look like.

## 3. A tool is an application capability with a contract

In agent systems, the word *tool* can sound more mysterious than it is. A tool is usually an ordinary function, API endpoint, database operation, or service that the application has chosen to expose through a controlled interface.

A useful tool definition needs more than executable code. It also needs a stable name, a description that explains when the tool is appropriate, an input contract, an output contract, and an execution policy. The policy matters because tools have different risk profiles. Reading public documentation is not equivalent to sending an email, transferring money, changing permissions, or deleting a record.

We will start with a deliberately low-risk arithmetic function.

In [ ]:
def multiply(first_number: float, second_number: float) -> float:
    """Multiply two numbers and return the numeric result."""

    return first_number * second_number


direct_result = multiply(first_number=145, second_number=28)
print(f"Direct function result: {direct_result}")

Direct function result: 4060


The calculator function works, but the model-like component and the tool are still unrelated. The model cannot communicate a tool choice by returning an informal sentence such as “perhaps calculate these numbers.” Application code needs an unambiguous structure that can be inspected and validated before anything is executed.

## 4. Represent the proposed action as structured data

A tool request should state which capability is being requested and which arguments should be supplied. In this first notebook, a Python dictionary is sufficient to represent that request.

This dictionary is only data. Creating it does not execute the tool, and the presence of a tool name does not grant permission to call an arbitrary Python function. Separating the proposal from its execution gives the application an opportunity to validate names, argument types, permissions, budgets, and approval requirements.

In [ ]:
tool_request = {
    "type": "tool_request",
    "tool_name": "multiply",
    "arguments": {
        "first_number": 145,
        "second_number": 28,
    },
}

print("Proposed action:")
print(tool_request)

Proposed action:
{'type': 'tool_request', 'tool_name': 'multiply', 'arguments': {'first_number': 145, 'second_number': 28}}


A production system would normally validate this structure with a formal schema. We will introduce Pydantic later in the course because it can reject missing fields, invalid types, unsupported action names, and inconsistent combinations before execution. For now, the dictionary keeps the control flow visible.

## 5. Build a registry rather than resolving arbitrary names

A naive implementation might attempt to convert whatever name the model produces into an executable Python object. That creates an unsafe and difficult-to-audit boundary. A safer design maintains an explicit registry containing only the capabilities this application intends to expose.

The registry below is an allowlist. If a requested name is absent, the application rejects the request regardless of how confidently the model produced it.

In [ ]:
tool_registry = {
    "multiply": multiply,
}

print("Registered tools:", list(tool_registry))

Registered tools: ['multiply']


The dictionary maps a public tool name to the Python function that implements it. In a more mature implementation, each registry entry would also contain a description, an argument schema, side-effect metadata, timeout settings, and an authorization policy. The underlying principle remains the same: the application defines the available action space.

## 6. Introduce the execution boundary

The executor is the point at which proposed data may become a real action. This boundary should be explicit because it is where validation, authorization, logging, timeout handling, and error conversion belong.

The implementation below performs three useful tasks. It checks the allowlist, executes the selected function with named arguments, and converts both success and failure into structured observations. Returning failures as observations allows the surrounding control loop to reason about recoverable problems instead of crashing immediately.

In [ ]:
from typing import Any, Callable


ToolFunction = Callable[..., Any]


def execute_tool_request(
    request: dict[str, Any],
    registry: dict[str, ToolFunction],
) -> dict[str, Any]:
    """Validate and execute one tool request, returning an observation."""

    tool_name = request.get("tool_name")
    arguments = request.get("arguments", {})

    if tool_name not in registry:
        return {
            "type": "tool_observation",
            "success": False,
            "tool_name": tool_name,
            "error": f"Tool is not registered: {tool_name}",
        }

    selected_tool = registry[tool_name]

    try:
        result = selected_tool(**arguments)
    except Exception as error:
        return {
            "type": "tool_observation",
            "success": False,
            "tool_name": tool_name,
            "error": f"{type(error).__name__}: {error}",
        }

    return {
        "type": "tool_observation",
        "success": True,
        "tool_name": tool_name,
        "result": result,
    }

In [ ]:
observation = execute_tool_request(tool_request, tool_registry)
print(observation)

{'type': 'tool_observation', 'success': True, 'tool_name': 'multiply', 'result': 4060}


The `**arguments` expression expands a dictionary into named function arguments. In this example, the executor effectively performs `multiply(first_number=145, second_number=28)`. This convention is widely used in tool-calling systems because JSON objects map naturally to named parameters.

Notice that the executor does not trust the request merely because its structure looks correct. The requested name must still exist in the registry. The next example demonstrates the rejection path.

In [ ]:
untrusted_request = {
    "type": "tool_request",
    "tool_name": "delete_all_files",
    "arguments": {},
}

rejected_observation = execute_tool_request(
    untrusted_request,
    tool_registry,
)

print(rejected_observation)

{'type': 'tool_observation', 'success': False, 'tool_name': 'delete_all_files', 'error': 'Tool is not registered: delete_all_files'}


The request was rejected because the capability was not registered. No attempt was made to locate a Python function with a matching name. This is a foundational security property: model output is treated as untrusted input, not as executable authority.

## 7. Turn the tool result into an observation

In agent terminology, the tool result is commonly called an *observation* because it becomes new evidence available to the next decision. The model may have known that multiplication was required without knowing the correct result. After execution, the state contains information that was not previously available.

A complete tool-calling cycle therefore contains two distinct model decisions. The first response requests an action. The application executes that action and records an observation. A later model response uses the updated context to produce a final answer or request another action.

We will simulate the second decision with an ordinary function so that the separation remains visible.

In [ ]:
def formulate_final_answer(
    user_question: str,
    tool_observation: dict[str, Any],
) -> str:
    """Create a user-facing answer from a structured observation."""

    if not tool_observation["success"]:
        return (
            "I could not complete the request because the tool step failed. "
            f"Details: {tool_observation['error']}"
        )

    return (
        f"The requested calculation produced "
        f"{tool_observation['result']}."
    )


user_question = "What is 145 multiplied by 28?"
final_answer = formulate_final_answer(user_question, observation)

print(final_answer)

The requested calculation produced 4060.


The implementation is still deterministic, but the architecture now contains the same boundaries that appear in a real tool-using agent:

                    ┌──────────────────────────┐
                    │        User goal         │
                    └────────────┬─────────────┘
                                 │
                                 ▼
                    ┌──────────────────────────┐
                    │ Model proposes an action │
                    └────────────┬─────────────┘
                                 │
                                 ▼
                    ┌──────────────────────────┐
                    │ Application validates    │
                    │      the proposal        │
                    └────────────┬─────────────┘
                                 │
                                 ▼
                    ┌──────────────────────────┐
                    │ Application executes an  │
                    │      approved tool       │
                    └────────────┬─────────────┘
                                 │
                                 ▼
                    ┌──────────────────────────┐
                    │ Tool result becomes an   │
                    │       observation        │
                    └────────────┬─────────────┘
                                 │
                                 ▼
                    ┌──────────────────────────┐
                    │ Model receives the       │
                    │     updated context      │
                    └────────────┬─────────────┘
                                 │
                                 ▼
                    ┌──────────────────────────┐
                    │ Model answers or proposes│
                    │     another action       │
                    └──────────────────────────┘

The model is responsible for proposing what might be useful next. The application remains responsible for deciding what is possible and permitted.

## 8. Why this is not yet a complete agent

We have implemented one controlled action cycle, but several elements are still manually supplied. The tool request was written by us rather than produced by a model, the final response used a deterministic formatter, and there is no loop that can continue across multiple decisions.

A reusable agent needs at least five components:

1. **A model interface** that can return either a final answer or a structured action request.
2. **A tool registry** that defines the action space exposed by the application.
3. **State** containing messages, action requests, observations, and relevant metadata.
4. **A control loop** that repeatedly invokes the model and executes approved actions.
5. **Stop conditions** that terminate execution on success, failure, step count, elapsed time, cost, or an authorization boundary.

The word *agent* should therefore refer to the complete governed system, not merely to the language model inside it.

## 9. Trace the lifecycle explicitly

Execution traces are useful for debugging and later evaluation. A final answer alone does not reveal whether the system chose an appropriate tool, supplied correct arguments, recovered from an error, or repeated unnecessary actions.

The trace below records the essential events from our first interaction. Later notebooks will generate these events automatically and attach latency, token usage, and cost metadata.

In [ ]:
interaction_trace = [
    {
        "event": "user_message",
        "content": user_question,
    },
    {
        "event": "model_tool_request",
        "tool_name": tool_request["tool_name"],
        "arguments": tool_request["arguments"],
    },
    {
        "event": "tool_observation",
        "success": observation["success"],
        "result": observation.get("result"),
        "error": observation.get("error"),
    },
    {
        "event": "model_final_answer",
        "content": final_answer,
    },
]

for step_number, event in enumerate(interaction_trace, start=1):
    print(f"Step {step_number}: {event}")

Step 1: {'event': 'user_message', 'content': 'What is 145 multiplied by 28?'}
Step 2: {'event': 'model_tool_request', 'tool_name': 'multiply', 'arguments': {'first_number': 145, 'second_number': 28}}
Step 3: {'event': 'tool_observation', 'success': True, 'result': 4060, 'error': None}
Step 4: {'event': 'model_final_answer', 'content': 'The requested calculation produced 4060.'}


## 10. Engineering perspective: autonomy should be earned

An agent is not automatically better than a deterministic workflow. If every step is already known, ordinary application code is usually easier to test, cheaper to run, and more predictable. Agentic control becomes valuable when runtime evidence should influence which action happens next and enumerating every useful path in advance would be impractical.

Increased autonomy also increases the number of possible trajectories. That expands the testing surface and makes observability, budgets, permissions, and evaluation more important. A senior design question is therefore not “How can we add an agent?” but “Which decision genuinely benefits from model-driven choice, and what is the smallest safe action space for that decision?”

## 11. Exercise: add a second capability

Implement a `subtract` tool and register it alongside `multiply`. Then create a request that calculates `100 - 37`, execute the request through `execute_tool_request`, and confirm that the resulting observation contains `63`.

Do not call `subtract` directly in the exercise. The purpose is to practice the full request → validation → execution → observation path.

In [ ]:
def subtract(first_number: float, second_number: float) -> float:
    # Replace this line with your implementation.
    raise NotImplementedError


exercise_registry = {
    "multiply": multiply,
    # Register subtract here.
}

exercise_request = {
    "type": "tool_request",
    "tool_name": "subtract",
    "arguments": {
        "first_number": 100,
        "second_number": 37,
    },
}

# Execute the request through execute_tool_request and inspect the observation.

<details>
<summary><strong>Reveal one possible solution</strong></summary>

```python
def subtract(first_number: float, second_number: float) -> float:
    return first_number - second_number


exercise_registry = {
    "multiply": multiply,
    "subtract": subtract,
}

exercise_request = {
    "type": "tool_request",
    "tool_name": "subtract",
    "arguments": {
        "first_number": 100,
        "second_number": 37,
    },
}

exercise_observation = execute_tool_request(
    exercise_request,
    exercise_registry,
)

print(exercise_observation)
```

The successful observation should contain `"result": 63`.

</details>

## 12. Review questions

Try to answer these questions without returning to the earlier sections:

1. Why is a tool request data rather than an executed action?
2. What security property does the registry provide?
3. Why should tool failures be converted into observations?
4. What information would you add to the interaction trace in a production system?
5. When would a deterministic workflow be preferable to an agent?

If you can explain each answer in your own words, you have the conceptual foundation required for the next notebook.

## Notebook summary

We began with a model-like text function and a separate arithmetic capability. We represented an action request as structured data, placed executable capabilities behind an explicit registry, and introduced a controlled execution boundary that converts both success and failure into observations. Finally, we recorded the lifecycle as a trace and identified the components still required for a reusable agent.

The next notebook will distinguish an LLM call, a chain, a deterministic workflow, a router, and an agent. Those categories are frequently blurred in discussions of Agentic AI, but understanding their differences prevents unnecessary complexity and leads to better system design.